# exp_6b — Experiment 6B: Semantic Image Segmentation Using Attention U-Net

Source-based notebook converted from the uploaded experiment code document.


**Note:** The uploaded source contains training code for Attention U-Net, but does not include a separate Mean IoU evaluation cell for Attention U-Net. The U-Net Mean IoU evaluation is present in Experiment 6A.


In [ ]:
import os

print("Images file exists:", os.path.exists(r"D:\DL DATASET\images.tar.gz"))
print("Annotations file exists:", os.path.exists(r"D:\DL DATASET\annotations.tar.gz"))

print("Images size (MB):", os.path.getsize(r"D:\DL DATASET\images.tar.gz") / (1024*1024))
print("Annotations size (MB):", os.path.getsize(r"D:\DL DATASET\annotations.tar.gz") / (1024*1024))

In [ ]:
import tarfile

try:
    with tarfile.open(r"D:\DL DATASET\images.tar.gz", "r:gz") as tar:
        members = tar.getmembers()
        print("Archive is valid.")
        print("Total files:", len(members))
except Exception as e:
    print("Archive is corrupted.")
    print(e)

In [ ]:
import tarfile
import os

os.makedirs("D:\DL DATASET", exist_ok=True)

with tarfile.open("D:\DL DATASET\images.tar.gz", "r:*") as tar:
    tar.extractall("D:\DL DATASET")

print("Images extracted successfully!")

In [ ]:
import tarfile
import os

extract_path = r"D:\DL DATASET"

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(r"D:\DL DATASET\annotations.tar.gz", "r:*") as tar:
    tar.extractall(extract_path)

print("Annotations extracted successfully!")

In [ ]:
import os

print("Images Folder:")
for root, dirs, files in os.walk(r"D:\DL DATASET\images"):
    print(root)
    print(files[:5])
    print("Total files:", len(files))
    print("-"*50)

print("\nAnnotations Folder:")
for root, dirs, files in os.walk(r"D:\DL DATASET\annotations"):
    print(root)
    print(files[:5])
    print("Total files:", len(files))
    print("-"*50)

In [ ]:
import os

print(os.path.exists(r"D:\DL DATASET\annotations.tar.gz"))

In [ ]:
import os

annotation_path = r"D:\DL DATASET\annotations"

print("Exists:", os.path.exists(annotation_path))
print("Folders:", os.listdir(annotation_path))

In [ ]:
for root, dirs, files in os.walk(r"D:\DL DATASET\annotations"):
    print(root)
    print(files[:5])
    print("-" * 50)

In [ ]:
import os
from glob import glob

IMAGE_DIR =  r"D:\DL DATASET\images"
MASK_DIR = r"D:\DL DATASET\annotations\trimaps"

# Get all image paths
image_paths = sorted(glob(os.path.join(IMAGE_DIR, "*.jpg")))

# Ignore hidden files like ._Bengal_58.png
mask_paths = sorted([
    f for f in glob(os.path.join(MASK_DIR, "*.png"))
    if not os.path.basename(f).startswith("._")
])

print("Total Images :", len(image_paths))
print("Total Masks  :", len(mask_paths))

print("\nSample Image :", image_paths[0])
print("Sample Mask  :", mask_paths[0])

In [ ]:
from sklearn.model_selection import train_test_split

train_images, test_images, train_masks, test_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42
)

print("Training Images :", len(train_images))
print("Testing Images  :", len(test_images))

In [ ]:
import tensorflow as tf

IMG_SIZE = 128
BATCH_SIZE = 16

def load_image_mask(img_path, mask_path):

    # Read image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    # Read mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")

    # Oxford-IIIT trimap values are {1,2,3}
    mask = tf.cast(mask, tf.int32) - 1

    return img, mask

In [ ]:
def create_dataset(images, masks):

    dataset = tf.data.Dataset.from_tensor_slices((images, masks))

    dataset = dataset.map(
        load_image_mask,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_dataset = create_dataset(train_images, train_masks)
test_dataset = create_dataset(test_images, test_masks)

print("Datasets created successfully!")

In [ ]:
import matplotlib.pyplot as plt

for image, mask in train_dataset.take(1):

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(image[0])
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(tf.squeeze(mask[0]), cmap="gray")
    plt.title("Segmentation Mask")
    plt.axis("off")

    plt.show()

In [ ]:
for images, masks in train_dataset.take(1):

    print("Image Shape :", images.shape)
    print("Mask Shape  :", masks.shape)

    unique = tf.unique(tf.reshape(masks, [-1]))[0]

    print("Unique Labels :", unique.numpy())

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

def conv_block(x, filters):

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    return x


def encoder_block(x, filters):

    c = conv_block(x, filters)
    p = MaxPooling2D((2,2))(c)

    return c, p


def decoder_block(x, skip, filters):

    x = Conv2DTranspose(filters,
                        kernel_size=2,
                        strides=2,
                        padding="same")(x)

    x = Concatenate()([x, skip])

    x = conv_block(x, filters)

    return x

In [ ]:
import tensorflow as tf

def dice_loss(y_true, y_pred, smooth=1e-6):

    y_true = tf.squeeze(y_true, axis=-1)

    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=3)

    y_true = tf.cast(y_true, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1,2]
    )

    union = tf.reduce_sum(
        y_true + y_pred,
        axis=[1,2]
    )

    dice = (2.0 * intersection + smooth) / (union + smooth)

    return 1 - tf.reduce_mean(dice)

In [ ]:
ce_loss = tf.keras.losses.SparseCategoricalCrossentropy()

def total_loss(y_true, y_pred):
    return ce_loss(y_true, y_pred) + dice_loss(y_true, y_pred)

In [ ]:
from tensorflow.keras.layers import *

def attention_gate(skip, gating, filters):

    theta_x = Conv2D(filters, 1, padding="same")(skip)
    theta_x = BatchNormalization()(theta_x)

    phi_g = Conv2D(filters, 1, padding="same")(gating)
    phi_g = BatchNormalization()(phi_g)

    add = Add()([theta_x, phi_g])
    add = Activation("relu")(add)

    psi = Conv2D(1, 1, padding="same")(add)
    psi = BatchNormalization()(psi)
    psi = Activation("sigmoid")(psi)

    out = Multiply()([skip, psi])

    return out

In [ ]:
def attention_decoder_block(x, skip, filters):

    x = Conv2DTranspose(
        filters,
        kernel_size=2,
        strides=2,
        padding="same"
    )(x)

    skip = attention_gate(skip, x, filters)

    x = Concatenate()([x, skip])

    x = conv_block(x, filters)

    return x

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D

inputs = Input(shape=(128,128,3))

# Encoder

s1,p1 = encoder_block(inputs,64)

s2,p2 = encoder_block(p1,128)

s3,p3 = encoder_block(p2,256)

s4,p4 = encoder_block(p3,512)

# Bottleneck

b = conv_block(p4,1024)

# Decoder

d1 = attention_decoder_block(b,s4,512)

d2 = attention_decoder_block(d1,s3,256)

d3 = attention_decoder_block(d2,s2,128)

d4 = attention_decoder_block(d3,s1,64)

outputs = Conv2D(
    3,
    kernel_size=1,
    activation="softmax"
)(d4)

attention_unet = Model(
    inputs,
    outputs,
    name="Attention_UNet"
)

attention_unet.summary()

In [ ]:
attention_unet.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss=total_loss,

    metrics=[
        "sparse_categorical_accuracy"
    ]

)

In [ ]:
attention_callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        verbose=1
    )

]

In [ ]:
history_attention = attention_unet.fit(

    train_dataset,

    validation_data=test_dataset,

    epochs=20,

    callbacks=attention_callbacks

)